# Lab 8: Correlation & Regression

In [ ]:
# Run this cell to set up the notebook, but please don't change it.
import numpy as np
from datascience import *

# These lines set up the plotting functionality and formatting.
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

# 1. The Dataset

In this lab, we are exploring movie screenplays. In particular, we have compiled a list of 5,000 words (including stemmed words) that occur in conversations between movie characters. For each movie, our dataset tells us the frequency with which each of these words occurs in certain conversations in its screenplay. All words have been converted to lowercase.

Run the cell below to read the `movies` table.

In [ ]:
movies = Table.read_table('/home/jovyan/shared/Data/movies.csv')
movies.where("Title", "wild wild west").select(0, 1, 2, 3, 4, 14, 49, 1042, 4004)

The above cell prints a few columns of the row for the comedy movie *Wild Wild West*.  The movie contains 3446 words. The word "it" appears 74 times, as it makes up  $\frac{74}{3446} \approx 0.021364$ of the words in the movie. The word "england" doesn't appear at all.
This numerical representation of a body of text, one that describes only the frequencies of individual words, is called a bag-of-words representation. A lot of information is discarded in this representation: the order of the words, the context of each word, who said what, the cast of characters and actors, etc. However, a bag-of-words representation is often used for machine learning applications as a reasonable starting point, because a great deal of information is also retained and expressed in a convenient and compact format.

All movie titles are unique. The `row_for_title` function provides access to the one row for each title. 

*Note: All movies in our dataset have their titles lower-cased.* 

In [ ]:
def row_for_title(title):
    """Return the row for a title"""
    
    return movies.where('Title', title).row(0)

For example, to find the frequency of "none" in the movie *The Terminator* is to access the `'none'` item from its row. Check the original table to see if this worked for you!

In [ ]:
row_for_title('the terminator').item('none') 

**Question 1.1.** Set `expected_row_sum` to the number that you __expect__ will result from summing all proportions in each row, excluding the first five columns.

In [ ]:
# Set row_sum to a number that's the (approximate) sum of each row of word proportions.
expected_row_sum = ...

This dataset was extracted from [a dataset from Cornell University](http://www.cs.cornell.edu/~cristian/Cornell_Movie-Dialogs_Corpus.html). After transforming the dataset (e.g., converting the words to lowercase, removing the naughty words, and converting the counts to frequencies), we created this new dataset containing the frequency of 5000 common words in each movie.

## 2. Word Stemming
The columns other than "Title", "Year", "Rating", "Genre", and "# Words" in the `movies` table are all words that appear in some of the movies in our dataset.  These words have been *stemmed*, or abbreviated heuristically, in an attempt to make different [inflected](https://en.wikipedia.org/wiki/Inflection) forms of the same base word into the same string.  For example, the column "manag" is the sum of proportions of the words "manage", "manager", "managed", and "managerial" (and perhaps others) in each movie. This is a common technique used in machine learning and natural language processing.

Stemming makes it a little tricky to search for the words you want to use, so we have provided another table that will let you see examples of unstemmed versions of each stemmed word.  Run the code below to load it.

In [ ]:
# Just run this cell.
vocab_mapping = Table.read_table('/home/jovyan/shared/Data/stem.csv')
stemmed = np.take(movies.labels, np.arange(5, len(movies.labels)))
vocab_table = Table().with_column('Stem', stemmed).join('Stem', vocab_mapping)
vocab_table.take(np.arange(1100, 1110))

**Question 2.1.** Assign `stemmed_message` to the stemmed version of the word "continuously".

In [ ]:
stemmed_message = ...
stemmed_message

**Question 2.2.** How many stems in `vocab_table` have eight or more words that are shortened to it? Assign that int value to `Eight_or_more_stem`.

In [ ]:
Eight_or_more_stem = ...
Eight_or_more_stem

**Question 2.3.** What is the word in `vocab_table` that was shortened the most to become the stem? Assign the string of that word to `longest_cut`. Break ties first by the longest original word length, and then reverse alphabetically from Z to A (so if your options are "albatross" or "blackbird", you should pick "blackbird").

In [ ]:
# In our solution, we found it useful to first add columns with
# the length of the word and the length of the stem,
# and then to add a column with the difference between those lengths.

tbl_with_lens = ...
tbl_with_dif = ...


longest_cut = ...
longest_cut

## 3. Relationships Between Pairs of Words

For this section we will look at the relationship between the proportion of the word `her` in screenplays versus the word `she`. We would like to use linear regression to make predictions about this relationship, but that won't work well if the data aren't roughly linearly related. To check that, we should look at the data. Run the cell below to construct a new table `she_her` that contains only the proportions of the words `she` and `her` in the screenplays.

In [ ]:
she_her = movies.select("she", "her")
she_her

**Question 3.1.** Make a scatter plot of the data.  It's conventional to put the column we want to predict on the vertical axis and the other column on the horizontal axis. Let's say we want to use the proportions of `she` to predict the proportions of `her`.

In [ ]:
...

**Question 3.2.** Are the proportions of `she` and `her` in the screenplays roughly linearly related based on the scatter plot above?

*Write your answer here, replacing this text.*

We're going to continue with the assumption that they are linearly related, so it's reasonable to use linear regression to analyze this data.

We'd next like to plot the data in standard units. If you don't remember the definition of standard units, textbook section [14.2](https://www.inferentialthinking.com/chapters/14/2/Variability.html#standard-units) might help!

**Question 3.3.** Compute the mean and standard deviation for the proportions of `she` and `her` in the screenplays.  **Then** create a table called `she_her_standard` containing the proportions of `she` and `her` in the screenplays in standard units.  The columns should be named `she (standard units)` and `her (standard units)`.

In [ ]:
she_mean = ...
she_std = ...
her_mean = ...
her_std = ...

she_her_standard = Table().with_columns(
    "she (standard units)", ...,
    "her (standard units)", ...
)

she_her_standard

**Question 3.4.** Plot the data again, but this time in standard units.

In [ ]:
...

You'll notice that this plot looks the same as the last one!  However, the data and axes are scaled differently.  So it's important to read the ticks on the axes.

**Question 3.5.** Which would you guess best describes the correlation between the proportions of `she` and `her` in this dataset?

1. correlation is positive (but not close to zero)
2. correlation is close to zero
3. correlation is negative (but not close to zero)

Assign `correlation` to the number corresponding to your guess.


In [ ]:
correlation = ...

**Question 3.6.** Compute the correlation `r_she_her`.  

*Hint:* Use `she_her_standard`.  Section [15.1](https://www.inferentialthinking.com/chapters/15/1/Correlation.html#calculating-r) explains how to do this.

In [ ]:
r_she_her = ...
r_she_her

## 4. The Regression Line
Correlation is the **slope of the regression line when the data are put in standard units**.

The next cell plots the regression line in standard units:

$$\text{her proportion in standard units} = r \times \text{she proportions in standard units}.$$

Then, it plots the data in standard units again, for comparison.

In [ ]:
def draw_line(slope=0, intercept=0, x=make_array(-1.5, 6), color='r'):
    y = x*slope + intercept
    plots.plot(x, y, color=color)

she_her_standard.scatter('she (standard units)', 'her (standard units)')
draw_line(slope = r_she_her)

How would you take a point in standard units and convert it back to original units?  We'd have to "stretch" its horizontal position by `she_std` and its vertical position by `her_std`. That means the same thing would happen to the slope of the line.

Stretching a line horizontally makes it less steep, so we divide the slope by the stretching factor.  Stretching a line vertically makes it more steep, so we multiply the slope by the stretching factor.

**Question 4.1.** Calculate the slope of the regression line in original units, and assign it to `slope_she_her`.

(If the "stretching" explanation is unintuitive, consult section [15.2](https://www.inferentialthinking.com/chapters/15/2/Regression_Line.html#the-equation-of-the-regression-line) in the textbook.)

In [ ]:
slope_she_her = ...
slope_she_her

We know that the regression line passes through the point `(she_mean, her_mean)`.  You might recall from high-school algebra that the equation for the line is therefore:

$$\text{her\_proportion} - \text{her\_mean} = \text{slope\_she\_her} \times (\text{she\_proportion} - \text{she\_mean}).$$

The rearranged equation becomes:

$$\text{her\_proportion} = \text{slope\_she\_her} \times \text{she\_proportion} + (- \text{slope\_she\_her} \times \text{she\_mean} + \text{her\_mean})$$


**Question 4.2.** Calculate the intercept in original units and assign it to `intercept_she_her`.

In [ ]:
intercept_she_her = ...
intercept_she_her

Below you should see a plot of the data in original units as well as the regression line. This plot should look the same as the one plotted at the beginning of this section, but with different values on the axes.

In [ ]:
she_her.scatter('she', 'her')
draw_line(slope = slope_she_her, intercept = intercept_she_her,  x = make_array(0,0.025))

You are done with Lab 8!